In [73]:
import pandas as pd
import numpy as np
import time
import os
from sklearn.preprocessing import MinMaxScaler

import matplotlib.pyplot as plt
import seaborn as sns
import yaml

with open("../config.yaml", "r") as file:
        config = yaml.safe_load(file)

output_dir = "../data/clean/"
config

In [74]:
df = pd.read_csv(config['output_data']['file1'], dtype={
        'code_insee_dept': str,
        'code_insee_comm': str,
        'zip_code': str
    })
df.head()

In [91]:
#Fill u p enmpty LIBDENS based on the comm_population col
def fill_up_empty_libdens(df):
    df1 = df.copy()

    mask = df1['LIBDENS'].isnull()
    df1.loc[mask & (df1['comm_population'] <= 2000), 'LIBDENS'] = "Rural"
    df1.loc[mask & ((df1['comm_population'] > 2000)
                   & (df1['comm_population'] <= 10000)), 'LIBDENS'] = "Intermediate urban"
    df1.loc[mask & (df1['comm_population'] > 10000), 'LIBDENS'] ="Dense urban"

    # translate 'comm_population' col to English
    df1['LIBDENS'] = df1['LIBDENS'].replace({'Urbain intermédiaire': "Intermediate urban", "Urbain dense":"Dense urban"})
    
   
    return df1
    
df1_urban = fill_up_empty_libdens(df)
df1_urban.head()

In [92]:
# Create latency categories
def latency_cat(df):
    df1 = fill_up_empty_libdens(df)
    df1 = df1.dropna(subset = ['avg_lat_down_ms', 'avg_lat_up_ms'])
    df1 = df1.drop(columns = 'zip_code')
    bins = [0, 20, 50, 100, 150, float('inf')]
    labels = ["Excellent", "Good", "Acceptable", "Fair", "Poor"]
    df1['load_lat_ms'] = (df1['avg_lat_down_ms'] + df1['avg_lat_up_ms'])/2
    df1["latency_category"] = pd.cut(
                                        df1['load_lat_ms'],
                                        bins = bins,
                                        labels = labels,
                                        right = False
                                      )
  
    return df1

df1 = latency_cat(df)
df1.head()

In [85]:
df1= fill_up_empty_libdens(df)
df1.shape # (229007, 21)
#df1.isna().sum()

In [94]:
#Broaddband quality score
def broadband_qty_score(df):
    df1 = latency_cat(df)
    #Normalize the load_lat_ms, avg_up_mbps and 	avg_down_mbps
    #create an instance oft he normalizer
    normalizer = MinMaxScaler()

    #fit the normalizer
    df1['down_speed_norm'] = normalizer.fit_transform(df1[['avg_down_mbps']])
    df1['up_speed_norm'] = normalizer.fit_transform(df1[['avg_up_mbps']])
    df1['load_lat_norm'] = 1 - normalizer.fit_transform(df1[['load_lat_ms']])

    #Calculate the borandband score
    df1['broadband_score%'] = round((df1['down_speed_norm']*0.5 + df1['up_speed_norm']*0.2 + df1['load_lat_norm']*0.3)*100, 2)

    return df1

df1 = broadband_qty_score(df)
df1.to_csv(f"{output_dir}fixed_internet_perf_clean.csv", index=False, encoding= "utf-8", sep = ",")
df1.head()
df1.duplicated().sum()
  

In [95]:
# defining fact and dimensions tables
def create_performance_tables(df):
    """
    create the fact table
    """
    df1 = broadband_qty_score(df)
    #create prefromance ID
    len_perf_tbl = len(df1)
    perf_id = [i for i in range(1, len_perf_tbl +1)]
        
    #Create performace table: with normalized data
    performance_table = df1[['avg_down_mbps',
                              'avg_up_mbps',
                              'avg_lat_ms',
                              'avg_lat_down_ms',
                              'avg_lat_up_ms',
                              'load_lat_ms',
                              'latency_category',
                              'nbr_tests',
                              'nbr_devices',
                              'down_speed_norm',
                              'up_speed_norm',
                              'load_lat_norm',
                              'broadband_score%',
                              'code_insee_region'
                             ]]
    performance_table.insert(0, 'perf_id', perf_id)
    
   #cretae region table
    region_table = df1[['code_insee_region', 
                    'region_name', 
                    'code_siren_region'
                   ]
                   ]
    region_table = region_table.drop_duplicates(subset = 'code_insee_region')
    
    #cretae department table
    department_table = df1[['code_insee_dept',
                         'department_name'
                        ]
                        ] 
    department_table = department_table.drop_duplicates(subset = 'code_insee_dept')

    #cretae department table
    commune_table = df1[['code_insee_comm',
                      'commune_name',
                      'commune_status',
                      'comm_population',
                      'superficie_cadastrale',
                      'LIBDENS'
                     ]]
    commune_table = commune_table.drop_duplicates(subset = 'code_insee_comm')
    commune_table = commune_table.reset_index(drop = True)
    
    return performance_table, region_table, department_table, commune_table               
                         
performance_table, region_table, department_table, commune_table = create_performance_tables(df)

performance_table.to_csv(f"{output_dir}performance_table.csv", index=False, encoding= "utf-8", sep = ",")
region_table.to_csv(f"{output_dir}region_table.csv", index=False, encoding= "utf-8", sep = ",")
department_table.to_csv(f"{output_dir}department_table.csv", index=False, encoding= "utf-8", sep = ",")
commune_table.to_csv(f"{output_dir}commune_table.csv", index=False, encoding= "utf-8", sep = ",")



In [96]:
commune_table